# E3' - LOCO (Leave-One-Center-Out) - BC-VMamba

Uji **generalisasi lintas-pusat** (kontribusi K2). Train di pusat lain, test pada pusat hold-out.
Desain (revisi kritik '7 centers'):
- **Primer:** leave-Cyprus-out, leave-Pisa_clin-out (population shift nyata).
- **Sekunder:** leave-ALL-technical-out (unseen acquisition, test=500).

Pipeline E0 identik E1a. Output `loco_{center}_test_per_image.csv` per pusat hold-out.
Batch antar-sesi: komentari baris di `LOCO_CONFIGS`.

---

# BC-VMamba-CUBS A1 — E1a READY (GT=A1 gold, patient-disjoint, save predictions)

# BC-VMamba-CUBS — Config A1 (Architecture-Only, Fair vs Baselines)

## Tujuan
Mengevaluasi **kontribusi arsitektur** BC-VMamba-CUBS dari panduan PDF (R1) secara terpisah dari
kontribusi loss/aug/aux-task. Notebook ini hanya mengganti **arsitektur** (TiFusion + MWFFD + EdgeAttention)
sementara loss, optimizer, scheduler, augmentasi, dataset, dan split **identik** dengan
`baselines_segmentation.ipynb` agar perbandingan apple-to-apple.

## Mapping ke PDF Panduan
- Konfigurasi : **Abl-A2** (PDF hal. 17) — BC-VMamba backbone (TiFusion) + MWFFD decoder
- Tidak diaktifkan di run ini: IMTGradeHead (Abl-A3), DBSHL boundary loss (Abl-A4),
  Speckle physics loss (Abl-A5), DDPM augmentation (Abl-B*), XAI in-training (Abl-C*).
- Hyperparameter mengikuti PDF Tabel R1-1 di titik-titik yang juga ada di baseline
  (AdamW, LR=1e-4, batch=8, image=256, Cosine, early-stop=15). Yang berbeda dengan PDF:
  - **Loss**: PDF resmi pakai DBSHL+Speckle (α·Dice + β·BCE + γ·Hausdorff + δ·Speckle).
    Di sini override jadi `SimpleLoss = 0.5·Dice + 0.5·CE(w=[1,3])` (loss baseline) supaya
    isolasi kontribusi arsitektur murni — lihat keterangan A1 di analisis.
  - **Output**: PDF resmi pakai sigmoid 1-channel + dual output. Di sini head di-set
    `Conv2d(c1, num_classes=2, 1)` tanpa sigmoid agar logits compatible langsung dengan
    `SimpleLoss` & `segmentation_metrics` baseline.
  - **Epochs**: 100 (convergence check sesuai baseline), bukan 60 fixed dari PDF.

## Yang Direuse Verbatim dari Baseline
Cell install, dataset pipeline, augmentasi (M3-UNet strong + mosaic p=0.3), preprocessing
(CLAHE + ROI crop + dataset-stats normalize), `SimpleLoss`, `segmentation_metrics`,
`measure_efficiency`, `train_one_epoch_baseline`, `evaluate_model_baseline`, `run_baseline_training`,
results CSV/plot.

## Run Setup
- BASELINE_NAMES = `["bc_vmamba_cubs_a1"]` (single config)
- K_FOLDS = 1 (convergence check 80/20), NUM_EPOCHS = 100, BATCH_SIZE = 8
- AMP enabled, grad clip 1.0, seed = 42


In [ ]:
import subprocess, sys, importlib

def run(cmd, show_output=False):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if show_output:
        print(result.stdout[-800:] if result.stdout else "")
    if result.returncode != 0:
        print("STDERR:", result.stderr[-500:])
    return result.returncode

def pkg_ok(name):
    try:
        importlib.import_module(name)
        return True
    except ImportError:
        return False

print("=" * 55)
print("STEP 0: Checking environment...")
run("nvidia-smi | head -10", show_output=True)
run("pip list 2>/dev/null | grep -i torch | head -5", show_output=True)

print("=" * 55)
print("STEP 1: PyTorch cu126...")
if not pkg_ok("torch"):
    print("  Installing (~5 menit)...")
    rc = run("pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126")
    print(f"  PyTorch cu126: {'OK' if rc == 0 else 'FAILED'}")
else:
    import torch
    print(f"  Already installed: {torch.__version__} -- SKIP")

print("=" * 55)
print("STEP 2: causal_conv1d...")
if not pkg_ok("causal_conv1d"):
    rc = run("pip install -q causal-conv1d>=1.4.0 --no-build-isolation")
    if rc != 0:
        rc = run("pip install -q causal-conv1d --no-build-isolation")
    print(f"  causal_conv1d: {'OK' if rc == 0 else 'FAILED'}")
else:
    print("  Already installed -- SKIP")

print("=" * 55)
print("STEP 3: mamba_ssm...")
if not pkg_ok("mamba_ssm"):
    _net = run("python -c \"import urllib.request as u; u.urlopen('https://pypi.org', timeout=8)\"")
    if _net != 0:
        raise RuntimeError("TIDAK ADA INTERNET. Panel kanan Kaggle: Settings > Internet = On (butuh akun terverifikasi telepon), lalu run ulang. mamba-ssm tak bisa diunduh tanpa internet.")
    rc = run("pip install -q mamba-ssm --no-build-isolation", show_output=True)
    if rc != 0:
        rc = run("pip install -q mamba-ssm --no-deps --no-build-isolation", show_output=True)
    print(f"  mamba_ssm: {'OK' if rc == 0 else 'FAILED'}")
else:
    print("  Already installed -- SKIP")

print("=" * 55)
print("STEP 4: Optional (thop, timm)...")
if not pkg_ok("thop"):
    run("pip install -q thop timm")
    print("  thop + timm: installed")
else:
    print("  thop + timm already installed -- SKIP")

print("=" * 55)
print("STEP 5: Verifying mamba_ssm import...")
v = subprocess.run([sys.executable, "-c", "from mamba_ssm import Mamba; print('mamba_ssm OK')"],
                   capture_output=True, text=True)
if v.returncode == 0:
    print(" ", v.stdout.strip())
    print("\n[OK] Semua dependency siap. Lanjutkan ke cell berikutnya.")
else:
    raise RuntimeError(
        "mamba_ssm gagal diimpor setelah instalasi. Error asli:\n"
        + (v.stderr[-1500:] or "(kosong)")
        + "\n\nCek: (1) Internet=On, (2) Accelerator=GPU (bukan None/CPU), "
          "(3) versi torch Kaggle vs wheel mamba-ssm di STEP 3 di atas.")


In [ ]:
# ============================================================
# MODELS BC-VMamba-CUBS A1 — Architecture-Only Config
# Writes models/bcvmamba_cubs.py and models/__init__.py
# Reuses same pattern as baselines_segmentation.ipynb cell 2 so get_model() works.
# ============================================================
import os
os.makedirs("models", exist_ok=True)

with open("models/__init__.py", "w", encoding="utf-8") as _f:
    _f.write(
"""# Auto-generated by bcvmamba_cubs_a1.ipynb — exposes single architecture.
from .bcvmamba_cubs import BCVMambaCUBS_A1

_DEFAULTS = {
    "bc_vmamba_cubs_a1": dict(base_ch=32, d_state=16),
}
AVAILABLE = list(_DEFAULTS.keys())

def get_model(name: str, num_classes: int = 2, **kwargs):
    if name not in _DEFAULTS:
        raise ValueError(f"Unknown model {name!r}. Available: {AVAILABLE}")
    params = {**_DEFAULTS[name], **kwargs}
    return BCVMambaCUBS_A1(num_classes=num_classes, **params)
""")

with open("models/bcvmamba_cubs.py", "w", encoding="utf-8") as _f:
    _f.write(
'''"""
BC-VMamba-CUBS — Config A1 (architecture-only).

Komponen (sesuai PDF R1):
  - VSSBlock        : bidirectional Mamba SSM (O(N)) untuk long-range boundary context
  - TiFusionModule  : local depthwise-separable conv (speckle) + VSSBlock (global) + channel gate
  - EdgeAttention   : Sobel-guided attention untuk fokus tepi LIB/MAB
  - MWFFDDecoder    : boundary-aware decoder dengan learnable weight skip/upsample + EdgeAttention
  - BCVMambaCUBS_A1 : U-shape encoder-decoder, head = Conv2d(c1, num_classes, 1) tanpa sigmoid
                       sehingga compatible dengan SimpleLoss(softmax)+segmentation_metrics(argmax) baseline.

Catatan implementasi:
  - TiFusion HANYA dipakai di level dalam (64x64 dan 32x32) + bottleneck (16x16) untuk menjaga
    memori T4 16GB pada batch=8. Top-2 level (256, 128) pakai ConvBlock saja: jumlah token Mamba
    di level 256x256 = 65536 dengan dim=32 akan memboroskan VRAM tanpa kontribusi besar
    pada konteks long-range. Ini konsisten dengan praktik VM-UNet & M3-UNet (Mamba di deeper layers).
  - VSSBlock pakai 2-way scan (forward + flipped) — implementasi paling umum di VSS literature.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from mamba_ssm import Mamba
except ImportError as _e:
    raise ImportError("mamba_ssm not installed. Run: pip install mamba-ssm causal-conv1d") from _e


# ----- Building blocks --------------------------------------------------------
class ConvBlock(nn.Module):
    """Plain double-conv block used di top-2 encoder levels (top-resolution)."""
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU(),
        )
    def forward(self, x): return self.block(x)


class VSSBlock(nn.Module):
    """Bidirectional Mamba SSM untuk 2D feature map (O(N) long-range modeling).

    forward + flipped scan (2-way) sepanjang flatten(H,W) sequence.
    Output di-proyeksikan kembali ke dim semula + residual.
    """
    def __init__(self, dim: int, d_state: int = 16, d_conv: int = 4, expand: int = 2):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.mamba_f = Mamba(d_model=dim, d_state=d_state, d_conv=d_conv, expand=expand)
        self.mamba_b = Mamba(d_model=dim, d_state=d_state, d_conv=d_conv, expand=expand)
        self.proj = nn.Linear(2 * dim, dim)
        self.skip_scale = nn.Parameter(torch.ones(1))

    def forward(self, x):
        if x.dtype == torch.float16:
            x = x.float()
        B, C, H, W = x.shape
        x_flat = x.flatten(2).transpose(1, 2)         # (B, HW, C)
        x_norm = self.norm(x_flat)
        y_f = self.mamba_f(x_norm)
        y_b = self.mamba_b(x_norm.flip(1)).flip(1)
        y   = self.proj(torch.cat([y_f, y_b], dim=-1)) + self.skip_scale * x_flat
        return y.transpose(1, 2).reshape(B, C, H, W)


class TiFusionModule(nn.Module):
    """Local depthwise-separable CNN (speckle) + Global VSSBlock + channel attention gate.

    PDF: f_fused = gate(concat(F_local, F_global)). Projected back ke ch dim.
    """
    def __init__(self, ch: int, d_state: int = 16, r: int = 4):
        super().__init__()
        # Local branch: depthwise separable conv (speckle texture)
        self.local = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, groups=ch, bias=False),   # depthwise
            nn.Conv2d(ch, ch, 1, bias=False),                          # pointwise
            nn.BatchNorm2d(ch), nn.GELU(),
        )
        # Global branch: VSS Mamba
        self.global_mamba = VSSBlock(ch, d_state=d_state)
        # Channel attention gate over [F_local; F_global]
        hid = max(1, (ch * 2) // r)
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(ch * 2, hid, 1), nn.ReLU(inplace=True),
            nn.Conv2d(hid, ch * 2, 1), nn.Sigmoid(),
        )
        self.proj = nn.Conv2d(ch * 2, ch, 1)

    def forward(self, x):
        fl  = self.local(x)
        fg  = self.global_mamba(x)
        cat = torch.cat([fl, fg], dim=1)
        return self.proj(cat * self.gate(cat))


class EdgeAttention(nn.Module):
    """Sobel-guided attention untuk fokus tepi LIB/MAB."""
    def __init__(self, ch: int):
        super().__init__()
        sx = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        self.register_buffer("sobel_x", sx)
        self.register_buffer("sobel_y", sx.transpose(-1, -2).contiguous())
        self.attn = nn.Sequential(nn.Conv2d(ch + 1, ch, 1), nn.Sigmoid())

    def forward(self, x):
        gray = x.mean(1, keepdim=True)
        ex   = F.conv2d(gray, self.sobel_x, padding=1).abs()
        ey   = F.conv2d(gray, self.sobel_y, padding=1).abs()
        edge = ex + ey
        return x * self.attn(torch.cat([x, edge], dim=1))


class MWFFDDecoder(nn.Module):
    """Mixed-Weight Fine Feature Decoder.
    Learnable scalar weight per stream (skip vs upsampled), refine conv, lalu EdgeAttention.
    """
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.w_sk = nn.Parameter(torch.ones(1))
        self.w_up = nn.Parameter(torch.ones(1))
        self.refine = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 1),
        )
        self.edge_attn = EdgeAttention(out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if skip.shape[-2:] != x.shape[-2:]:
            skip = F.interpolate(skip, size=x.shape[-2:], mode="bilinear", align_corners=False)
        fused = torch.cat([self.w_up * x, self.w_sk * skip], dim=1)
        return self.edge_attn(self.refine(fused))


# ----- Full model -------------------------------------------------------------
class BCVMambaCUBS_A1(nn.Module):
    """A1 config: arsitektur saja, output logits 2-channel compatible dengan SimpleLoss baseline.

    Encoder channel layout: [base, base*2, base*4, base*8] + bottleneck base*16.
    Default base=32 -> [32, 64, 128, 256, 512]. Total ~9-15M params (cek di sanity).
    """
    def __init__(self, in_ch: int = 1, num_classes: int = 2, base_ch: int = 32, d_state: int = 16):
        super().__init__()
        c1, c2, c3, c4 = base_ch, base_ch * 2, base_ch * 4, base_ch * 8
        cb             = base_ch * 16

        # ---------------- Encoder ----------------
        self.stem  = nn.Conv2d(in_ch, c1, 3, padding=1)
        # Level 1 & 2: ConvBlock saja (top resolution, Mamba terlalu boros memori)
        self.enc1  = ConvBlock(c1, c1)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), nn.Conv2d(c1, c2, 1, bias=False))
        self.enc2  = ConvBlock(c2, c2)
        self.down2 = nn.Sequential(nn.MaxPool2d(2), nn.Conv2d(c2, c3, 1, bias=False))
        # Level 3 & 4: TiFusion (local + global Mamba)
        self.enc3  = TiFusionModule(c3, d_state=d_state)
        self.down3 = nn.Sequential(nn.MaxPool2d(2), nn.Conv2d(c3, c4, 1, bias=False))
        self.enc4  = TiFusionModule(c4, d_state=d_state)
        self.down4 = nn.Sequential(nn.MaxPool2d(2), nn.Conv2d(c4, cb, 1, bias=False))

        # ---------------- Bottleneck ----------------
        self.bottleneck = TiFusionModule(cb, d_state=d_state)

        # ---------------- Decoder ----------------
        self.dec4 = MWFFDDecoder(cb, c4, c4)
        self.dec3 = MWFFDDecoder(c4, c3, c3)
        self.dec2 = MWFFDDecoder(c3, c2, c2)
        self.dec1 = MWFFDDecoder(c2, c1, c1)

        # ---------------- Segmentation head (logits, no sigmoid) ----------------
        self.seg_head = nn.Conv2d(c1, num_classes, 1)

        print(f"  [BCVMambaCUBS_A1] base_ch={base_ch}, d_state={d_state}, num_classes={num_classes}")

    def forward(self, x):
        x  = self.stem(x)                       # (B, c1,  H,    W)
        e1 = self.enc1(x)                       # (B, c1,  H,    W)
        e2 = self.enc2(self.down1(e1))          # (B, c2,  H/2,  W/2)
        e3 = self.enc3(self.down2(e2))          # (B, c3,  H/4,  W/4)
        e4 = self.enc4(self.down3(e3))          # (B, c4,  H/8,  W/8)
        b  = self.bottleneck(self.down4(e4))    # (B, cb,  H/16, W/16)
        d4 = self.dec4(b,  e4)                  # (B, c4,  H/8,  W/8)
        d3 = self.dec3(d4, e3)                  # (B, c3,  H/4,  W/4)
        d2 = self.dec2(d3, e2)                  # (B, c2,  H/2,  W/2)
        d1 = self.dec1(d2, e1)                  # (B, c1,  H,    W)
        return self.seg_head(d1)                # (B, num_classes, H, W)  -- logits
''')

print("models/bcvmamba_cubs.py and models/__init__.py written.")


In [ ]:
import os, csv, json, math, time, random, warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2
from timm.models.layers import trunc_normal_
from scipy.ndimage import distance_transform_edt
from sklearn.model_selection import KFold

try:
    from mamba_ssm import Mamba
    MAMBA_AVAILABLE = True
    print("mamba_ssm loaded successfully.")
except Exception as e:
    MAMBA_AVAILABLE = False
    print("WARNING: mamba_ssm not available — vm_unet_baseline tidak bisa dijalankan (OK untuk sesi 1 & 2).")

try:
    from thop import profile as thop_profile
    THOP_AVAILABLE = True
except Exception:
    THOP_AVAILABLE = False

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

AMP_DEVICE_TYPE = "cuda" if device.type == "cuda" else "cpu"
USE_AMP    = (device.type == "cuda")
PIN_MEMORY = (device.type == "cuda")

print(f"Device  : {device}")
print(f"AMP     : {USE_AMP} ({AMP_DEVICE_TYPE})")
print(f"THOP    : {THOP_AVAILABLE}")
print(f"Torch   : {torch.__version__}")


import sys
sys.path.insert(0, './models')
print('models path set')

In [ ]:
# ============================================================
# KONFIGURASI BC-VMamba-CUBS A1 (Architecture-Only)
# Hyperparameter identik baselines_segmentation.ipynb -> apple-to-apple comparison.
# ============================================================
BASELINE_NAMES = ["bc_vmamba_cubs_a1"]   # single config; loop tetap dipakai untuk reuse driver baseline
LAMBDA_CIMT    = 0.2     # bobot L_cimt. HARUS sama dengan E1c: LOCO menguji model USULAN,
                         # bukan model base. 0 -> region-only (jangan dipakai untuk paper).

IMG_SIZE       = 256     # PDF: 256x256 = baseline
K_FOLDS        = 5       # 5-fold cross-validation (final reporting). Set 1 utk convergence check.
SEED           = 42
BATCH_SIZE     = 8       # PDF Tabel R1-1: 8 (T4x2)
NUM_WORKERS    = 2
NUM_EPOCHS     = 100     # baseline convergence check; PDF resmi 60 (early stop @15) -- override agar fair
WARMUP_EPOCHS  = 5
LR             = 1e-4    # PDF & baseline
WEIGHT_DECAY   = 1e-4
ETA_MIN        = 1e-6
EARLY_STOP_PAT = 15      # PDF & baseline

# ============================================================
# Environment & Path Detection (identik baseline cell 5)
# ============================================================
import os
from pathlib import Path

_ON_KAGGLE = Path('/kaggle/input').exists()
_ON_COLAB  = Path('/content').exists() and not _ON_KAGGLE

if _ON_COLAB:
    if not Path('/content/drive/MyDrive').exists():
        from google.colab import drive
        drive.mount('/content/drive')
    _DRIVE     = Path('/content/drive/MyDrive')
    _CUBS_BASE = _DRIVE / "CUBS"

    def _find(pat):
        hits = sorted(_CUBS_BASE.glob(pat))
        if hits: return hits[0]
        for _s in sorted(_CUBS_BASE.iterdir()) if _CUBS_BASE.exists() else []:
            if _s.is_dir():
                hits = sorted(_s.glob(pat))
                if hits: return hits[0]
        return None

    _cubs21_dir = (_find("DATASET for Carotid*") or _find("DATASET_CUBS_clin*")
                   or _CUBS_BASE / "DATASET for Carotid Ultrasound Boundary Study CUBS an open multi-center analysis of computerized intima-media thickness measurement systems and their clinical impact")
    _cubs22_dir = _find("DATASET_CUBS_tech") or _CUBS_BASE / "DATASET_CUBS_tech"
    _split_path = _find("split_info.json")   or _CUBS_BASE / "split_info.json"
    OUTPUT_BASE = _DRIVE / "bcvmamba_a1_results"
    print(f"Colab Drive paths (CUBS_BASE={_CUBS_BASE}):")
elif _ON_KAGGLE:
    def _kfind(pat):
        hits = sorted(Path('/kaggle/input').rglob(pat))
        return hits[0] if hits else None
    _cubs21_dir = _kfind("DATASET for Carotid*")
    _cubs22_dir = _kfind("DATASET_CUBS_tech")
    _split_path = _kfind("split_info.json")
    OUTPUT_BASE = Path("./results/bcvmamba_a1")
    print(f"Kaggle paths: cubs21={_cubs21_dir}, cubs22={_cubs22_dir}")
else:
    _LOCAL      = Path("../dataset")
    _cubs21_dir = _LOCAL / "DATASET for Carotid Ultrasound Boundary Study CUBS an open multi-center analysis of computerized intima-media thickness measurement systems and their clinical impact"
    _cubs22_dir = _LOCAL / "DATASET/CUBStech"
    _split_path = _LOCAL / "outputs/split_info.json"
    OUTPUT_BASE = Path("./results/bcvmamba_a1")
    print("Local mode")

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
KAGGLE_INPUT = Path('/kaggle/input')

print(f"\nModel      : {BASELINE_NAMES}")
print(f"Epochs     : {NUM_EPOCHS}  (warmup={WARMUP_EPOCHS}, early_stop_pat={EARLY_STOP_PAT})")
print(f"K_FOLDS    : {K_FOLDS}  ({'convergence check 80/20 split' if K_FOLDS==1 else 'final CV'})")
print(f"LR / Batch : {LR} / {BATCH_SIZE}  (AdamW, weight_decay={WEIGHT_DECAY})")
print(f"Loss       : SimpleLoss = 0.5*Dice + 0.5*CE(weight=[1,3]) + {LAMBDA_CIMT}*L_cimt")
print(f"Output     : {OUTPUT_BASE.resolve()}")
for _lbl, _p in [("cubs21", _cubs21_dir), ("cubs22", _cubs22_dir), ("split", _split_path)]:
    _st = "OK" if _p is not None and Path(str(_p)).exists() else "NOT FOUND"
    print(f"  {_st}  {_lbl}: {_p}")


In [ ]:
# ================================================================= CELL 5 (pengganti)
import json, csv, warnings
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import pandas as pd
import cv2

GT_ANNOT = "Manual-A1"    # gold standard

CUBS21_DIR = Path(str(_cubs21_dir))
CUBS22_DIR = Path(str(_cubs22_dir))

def _find_file(fname):
    """Cari file (master_index.csv / splits_5fold.json) di Kaggle input atau cwd."""
    for base in [Path("."), Path("./data"), KAGGLE_INPUT if 'KAGGLE_INPUT' in globals() else Path("/kaggle/input")]:
        hit = list(Path(base).rglob(fname)) if Path(base).exists() else []
        if hit:
            return hit[0]
    raise FileNotFoundError(f"{fname} tidak ditemukan. Upload output E0 ke Kaggle.")

MASTER_CSV  = _find_file("master_index.csv")
SPLITS_JSON = _find_file("splits_5fold.json")
print(f"master_index: {MASTER_CSV}")
print(f"splits      : {SPLITS_JSON}")

master = pd.read_csv(MASTER_CSV)
META   = master.set_index("image_id").to_dict("index")   # id -> {center,snr,morph,release,...}

@dataclass
class FoldResult:
    fold: int; dice: float; iou: float; accuracy: float; recall: float
    precision: float; mae_cimt_mm: float; mse_cimt_mm: float
    params_m: float; gflops: float; latency_ms: float; max_mem_mb: float
    best_epoch: int = 0

# ---- resolusi path per image_id (portable: dari CUBS21_DIR/CUBS22_DIR, bukan path absolut CSV)
def resolve_paths(iid: str):
    if iid.startswith("clin_"):
        seg = CUBS21_DIR / "SEGMENTATIONS" / GT_ANNOT
        return dict(img=CUBS21_DIR/"IMAGES"/f"{iid}.tiff",
                    li=seg/f"{iid}-LI.txt", ma=seg/f"{iid}-MA.txt",
                    rect=seg/f"{iid}_rect.txt", cf=CUBS21_DIR/"CF"/f"{iid}_CF.txt")
    seg = CUBS22_DIR / "LIMA-Profiles" / GT_ANNOT
    return dict(img=CUBS22_DIR/"images"/f"{iid}.tiff",
                li=seg/f"{iid}-LI.txt", ma=seg/f"{iid}-MA.txt",
                rect=seg/f"{iid}_rect.txt", cf=CUBS22_DIR/"CF"/f"{iid}_CF.txt")

def load_calibration_factor(iid: str):
    try:
        return float(Path(resolve_paths(iid)["cf"]).read_text().strip().split()[0])
    except Exception:
        return None

# ---- split E0 -> cv_names, test_names, PATIENT_FOLD_SPLITS (patient-disjoint)
_split = json.load(open(SPLITS_JSON, encoding="utf-8"))
_pid2imgs = master.groupby("patient_id")["image_id"].apply(list).to_dict()

test_names = np.array([im for p in _split["test"] for im in _pid2imgs[p]])
_cv_list, _fold_tag = [], []
for k in [f"fold{i+1}" for i in range(5)]:
    for p in _split["folds"][k]:
        for im in _pid2imgs[p]:
            _cv_list.append(im); _fold_tag.append(k)
cv_names   = np.array(_cv_list)
_fold_tag  = np.array(_fold_tag)
PATIENT_FOLD_SPLITS = [(np.where(_fold_tag != k)[0], np.where(_fold_tag == k)[0])
                       for k in [f"fold{i+1}" for i in range(5)]]

print(f"cv_names   : {len(cv_names)}  (5-fold patient-disjoint)")
print(f"test_names : {len(test_names)}  (held-out)")
print(f"folds (val): {[len(v) for _, v in PATIENT_FOLD_SPLITS]}")


In [ ]:
# ================================================================= CELL 6 (pengganti)
import random, torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

def load_profile(txt):
    pts = []
    for line in Path(txt).read_text().splitlines():
        s = line.split()
        if len(s) >= 2:
            pts.append((float(s[0]), float(s[1])))
    return np.asarray(pts, dtype=np.float32)

def rasterize_imc(li, ma, hw):
    """Mask IMC = area antara batas LI (atas) & MA (bawah)."""
    H, W = hw
    if len(li) < 2 or len(ma) < 2:
        return np.zeros((H, W), np.uint8)
    li = li[np.argsort(li[:, 0])]
    ma = ma[np.argsort(ma[:, 0])][::-1]
    poly = np.vstack([li, ma]).astype(np.int32)
    m = np.zeros((H, W), np.uint8)
    cv2.fillPoly(m, [poly], 1)
    return m

def read_rect(rect_path):
    try:
        v = Path(rect_path).read_text().split()
        return float(v[0]), float(v[1]), float(v[2]), float(v[3])   # x y w h
    except Exception:
        return None

def crop_roi_with_padding(image, roi_x, roi_y, roi_w, roi_h, v_pad=1.0, h_pad=0.1):
    h, w = image.shape[:2]
    pad_v = roi_h * v_pad
    y0 = int(max(0, roi_y - pad_v / 2)); y1 = int(min(h, roi_y + roi_h + pad_v / 2))
    pad_h = roi_w * h_pad
    x0 = int(max(0, roi_x - pad_h));     x1 = int(min(w, roi_x + roi_w + pad_h))
    cropped = image[y0:y1, x0:x1]
    ch, cw = cropped.shape[:2]
    sq = max(ch, cw, 1)
    square = np.zeros((sq, sq), dtype=image.dtype)
    yo, xo = (sq - ch) // 2, (sq - cw) // 2
    square[yo:yo+ch, xo:xo+cw] = cropped
    return square, float(sq)

def compute_dataset_stats(names, image_size=256):
    s, sq, n = 0.0, 0.0, 0
    for name in tqdm(names, desc="mean/std", leave=False):
        p = resolve_paths(str(name))
        img = cv2.imread(str(p["img"]), cv2.IMREAD_GRAYSCALE)
        if img is None: continue
        rect = read_rect(p["rect"])
        if rect is not None:
            img, _ = crop_roi_with_padding(img, *rect)
        img = cv2.resize(img, (image_size, image_size))
        pix = img.astype(np.float64) / 255.0
        nz = pix > 0
        if nz.any():
            c = pix[nz]; s += c.sum(); sq += (c**2).sum(); n += c.size
    mean = s / max(1, n)
    return float(mean), float(max(np.sqrt(sq / max(1, n) - mean**2), 1e-6))

DATASET_MEAN, DATASET_STD = compute_dataset_stats(cv_names, IMG_SIZE)
print(f"DATASET_MEAN={DATASET_MEAN:.6f}, DATASET_STD={DATASET_STD:.6f}")

def _acoustic_shadow(img, factor=0.5):
    w = img.shape[1]; sw = max(1, random.randint(w//8, w//4)); sx = random.randint(0, max(0, w-sw))
    out = img.astype(np.float32).copy(); out[:, sx:sx+sw] = np.clip(out[:, sx:sx+sw]*factor, 0, 255)
    return out.astype(img.dtype)
def _acoustic_enh(img, factor=1.5):
    h = img.shape[0]; eh = max(1, random.randint(h//8, h//4)); ey = random.randint(0, max(0, h-eh))
    out = img.astype(np.float32).copy(); out[ey:ey+eh, :] = np.clip(out[ey:ey+eh, :]*factor, 0, 255)
    return out.astype(img.dtype)

def get_train_transform():
    return A.Compose([
        A.Affine(translate_percent={"x": (-0.2, 0.2), "y": (-0.2, 0.2)}, scale=(0.8, 1.2),
                 rotate=(-30, 30), shear=(-30, 30), border_mode=cv2.BORDER_CONSTANT, p=0.7),
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
        A.RandomResizedCrop(size=(256, 256), scale=(0.25, 1.0), ratio=(1.0, 1.0), p=0.3),
        A.ElasticTransform(alpha=10, sigma=5, border_mode=cv2.BORDER_CONSTANT, p=0.3),
        A.GridDistortion(num_steps=5, distort_limit=0.2, border_mode=cv2.BORDER_CONSTANT, p=0.3),
        A.Lambda(image=lambda img, **kw: _acoustic_shadow(img, 0.5), p=0.5),
        A.Lambda(image=lambda img, **kw: _acoustic_enh(img, 1.5), p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.5, contrast_limit=0.5, p=0.5),
        A.GaussNoise(std_range=(0.05, 0.1), mean_range=(0.0, 0.0), p=0.3),
        A.GaussianBlur(blur_limit=(3, 3), p=0.3), A.MedianBlur(blur_limit=3, p=0.3),
        A.MotionBlur(blur_limit=3, p=0.3), A.Equalize(p=0.3),
        A.Normalize(mean=(DATASET_MEAN,), std=(DATASET_STD,), max_pixel_value=255.0), ToTensorV2()])

def get_val_transform():
    return A.Compose([A.Normalize(mean=(DATASET_MEAN,), std=(DATASET_STD,), max_pixel_value=255.0),
                      ToTensorV2()])

def measure_imt_from_mask(binary_mask, calibration_factor, roi_square_side, img_size=256):
    """Column-wise mean CIMT (mm) — sama seperti versi lama (per kolom LI->MA)."""
    scale = roi_square_side / float(img_size)
    b = (binary_mask > 0.5)
    th = []
    for col in range(b.shape[1]):
        rows = np.where(b[:, col])[0]
        if len(rows) >= 2:
            th.append((rows[-1] - rows[0]) * scale * calibration_factor)
    return float(np.mean(th)) if th else 0.0

class CUBSCombinedDataset(Dataset):
    def __init__(self, names, image_size=256, train=True, mosaic_p=0.3):
        self.names = [str(n) for n in names]
        self.image_size = image_size
        self.train = train
        self.mosaic_p = mosaic_p if train else 0.0
        self.transform = get_train_transform() if train else get_val_transform()

    def _load_raw(self, name):
        p = resolve_paths(name)
        image = cv2.imread(str(p["img"]), cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise FileNotFoundError(f"Image not found: {p['img']}")
        li, ma = load_profile(p["li"]), load_profile(p["ma"])
        mask = rasterize_imc(li, ma, image.shape) * 255      # 0/255
        rect = read_rect(p["rect"])
        if rect is not None:
            image, sq = crop_roi_with_padding(image, *rect)
            mask,  _  = crop_roi_with_padding(mask,  *rect)
        else:
            h, w = image.shape[:2]; sq = float(max(h, w))
        return image, mask, sq

    def _make_mosaic(self, idx):
        half = self.image_size // 2
        idxs = [idx] + random.choices(range(len(self.names)), k=3)
        imgs, masks = [], []
        for i in idxs:
            img, msk, _ = self._load_raw(self.names[i])
            imgs.append(cv2.resize(img, (half, half)))
            masks.append(cv2.resize(msk, (half, half), interpolation=cv2.INTER_NEAREST))
        mi = np.concatenate([np.concatenate([imgs[0], imgs[1]], 1),
                             np.concatenate([imgs[2], imgs[3]], 1)], 0)
        mm = np.concatenate([np.concatenate([masks[0], masks[1]], 1),
                             np.concatenate([masks[2], masks[3]], 1)], 0)
        return mi, mm

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        if self.train and random.random() < self.mosaic_p:
            image, mask = self._make_mosaic(idx); sq = float(self.image_size)
        else:
            image, mask, sq = self._load_raw(name)
            image = cv2.resize(image, (self.image_size, self.image_size))
            mask  = cv2.resize(mask,  (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 127).astype(np.float32)
        aug = self.transform(image=image, mask=mask)
        cf = load_calibration_factor(name)
        cimt_gt = measure_imt_from_mask(aug["mask"].numpy().astype(np.float32),
                                        cf, float(sq), self.image_size) if cf else 0.0
        return {"image": aug["image"], "mask": aug["mask"].long(), "name": name,
                "roi_square_side": sq, "cimt_gt": torch.tensor(cimt_gt, dtype=torch.float32)}

def make_dataloaders(train_names, val_names):
    persistent = NUM_WORKERS > 0
    tr = DataLoader(CUBSCombinedDataset(train_names, IMG_SIZE, True),
                    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                    pin_memory=PIN_MEMORY, persistent_workers=persistent, drop_last=True)
    va = DataLoader(CUBSCombinedDataset(val_names, IMG_SIZE, False),
                    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                    pin_memory=PIN_MEMORY, persistent_workers=persistent)
    return tr, va

print("Dataset (A1-rasterized, patient-disjoint folds) ready.")


In [ ]:
# ================================================================
# LOCO FOLDS - leave-one-center-out (revisi kritik "7 centers dibesarkan")
# Primer  : leave-Cyprus-out & leave-Pisa_clin-out (population shift nyata, test besar)
# Sekunder: leave-ALL-technical-out (train=klinis 2176 -> test=teknikal 500; unseen acquisition)
# (LOCO per-pusat-teknikal mungil TIDAK dipakai: train hanya berubah -3.7% -> bukan domain shift.)
# val di-carve patient-disjoint dari train pool utk early stopping.
# ================================================================
_locorng = np.random.default_rng(SEED)
def _carve_val(pool_df, frac=0.12):
    pts = list(pool_df["patient_id"].unique())
    pts = [pts[i] for i in _locorng.permutation(len(pts))]
    nval = max(1, round(frac*len(pts))); vp = set(pts[:nval])
    val = pool_df[pool_df["patient_id"].isin(vp)]["image_id"].to_numpy()
    trn = pool_df[~pool_df["patient_id"].isin(vp)]["image_id"].to_numpy()
    return trn, val

LOCO_CONFIGS = [
    ("Cyprus",    (master["center"]  == "Cyprus")),
    ("Pisa_clin", (master["center"]  == "Pisa_clin")),
    ("AllTech",   (master["release"] == "technical")),
]  # jalankan subset di sesi berbeda dgn mengomentari baris

LOCO_FOLDS = []
for _nm, _mask in LOCO_CONFIGS:
    _test = master[_mask]; _pool = master[~_mask]
    _trn, _val = _carve_val(_pool)
    LOCO_FOLDS.append(dict(name=_nm, train_names=_trn, val_names=_val,
                           test_names=_test["image_id"].to_numpy()))
    print(f"LOCO leave-{_nm:10s}-out: train={len(_trn):4d} val={len(_val):3d} "
          f"test={len(_test):4d}  (pusat latih: {sorted(_pool['center'].unique())})")
print("LOCO_FOLDS siap:", [f["name"] for f in LOCO_FOLDS])


In [ ]:
from models import get_model, AVAILABLE

print("Sanity check semua baseline dalam BASELINE_NAMES:")
for _name in BASELINE_NAMES:
    m = get_model(_name, num_classes=2).to(device)
    m.eval()
    with torch.no_grad():
        dummy = torch.randn(1, 1, IMG_SIZE, IMG_SIZE, device=device)
        out   = m(dummy)
    assert out.shape == (1, 2, IMG_SIZE, IMG_SIZE), f"Wrong shape {_name}: {out.shape}"
    p = sum(x.numel() for x in m.parameters()) / 1e6
    print(f"  {_name:<22}: {p:.2f}M params — output {out.shape}  OK")
    del m, dummy, out
if device.type == 'cuda':
    torch.cuda.empty_cache()
print("Semua model OK.")


In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth: float = 1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1).clamp(1e-6, 1 - 1e-6)
        nc    = probs.shape[1]
        t_oh  = F.one_hot(targets.long(), nc).permute(0, 3, 1, 2).float()
        p_f   = probs.flatten(2); t_f = t_oh.flatten(2)
        intersection = (p_f * t_f).sum(-1)
        dice_idx = (2 * intersection + self.smooth) / (p_f.sum(-1) + t_f.sum(-1) + self.smooth)
        return 1 - dice_idx.mean()



def segmentation_metrics(logits, targets, eps=1e-6):
    preds   = torch.argmax(logits, dim=1)
    pred_fg = (preds == 1); true_fg = (targets == 1)
    tp = (pred_fg & true_fg).sum().float()
    fp = (pred_fg & ~true_fg).sum().float()
    fn = (~pred_fg & true_fg).sum().float()
    tn = (~pred_fg & ~true_fg).sum().float()
    return {
        "dice":      float((2*tp+eps)/(2*tp+fp+fn+eps)),
        "iou":       float((tp+eps)/(tp+fp+fn+eps)),
        "accuracy":  float((tp+tn+eps)/(tp+tn+fp+fn+eps)),
        "recall":    float((tp+eps)/(tp+fn+eps)),
        "precision": float((tp+eps)/(tp+fp+eps)),
    }



class ThicknessConsistencyLoss(nn.Module):
    """L_cimt (terdiferensiasi) — samakan KETEBALAN kolom prediksi vs GT.
    Disalin apa adanya dari E1c_bcvmamba_Lcimt_READY.ipynb (sel 8) supaya LOCO
    melatih model USULAN, bukan model base.
      t_c = sum_baris softmax_fg[:, :, c]   (ketebalan lunak per kolom, dalam px)
      g_c = sum_baris GT[:, :, c]
      L   = mean_c |t_c - g_c| / H          (dinormalisasi tinggi H -> ~[0,1])
    """
    def forward(self, logits, targets):
        p = torch.softmax(logits, dim=1)[:, 1]     # B,H,W  prob foreground
        t = p.sum(dim=1)                            # B,W    ketebalan lunak prediksi
        g = targets.float().sum(dim=1)              # B,W    ketebalan GT
        return (t - g).abs().mean() / p.shape[1]


class SimpleLoss(nn.Module):
    """0.5*Dice + 0.5*CE(weight=[1,3]) + LAMBDA_CIMT * L_cimt.
    LAMBDA_CIMT = 0 -> setara model base (region-only). Untuk paper harus 0.2."""
    def __init__(self, lam=None):
        super().__init__()
        self.dice = DiceLoss()
        ce_w = torch.tensor([1.0, 3.0], device=device)
        self.ce   = nn.CrossEntropyLoss(weight=ce_w)
        self.cimt = ThicknessConsistencyLoss()
        self.lam  = LAMBDA_CIMT if lam is None else lam

    def forward(self, logits, targets):
        base = 0.5 * self.dice(logits, targets) + 0.5 * self.ce(logits, targets.long())
        return base + self.lam * self.cimt(logits, targets)

print(f"SimpleLoss (+L_cimt, lambda={LAMBDA_CIMT}), DiceLoss, ThicknessConsistencyLoss ready.")


In [ ]:
def measure_efficiency(model, img_size=256):
    model.eval()
    params_m = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    gflops   = float("nan")
    if THOP_AVAILABLE:
        try:
            dummy   = torch.randn(1, 1, img_size, img_size, device=device)
            macs, _ = thop_profile(model, inputs=(dummy,), verbose=False)
            gflops  = float(2.0 * macs / 1e9)
        except Exception as e:
            warnings.warn(f"GFLOPs gagal: {e}")
    runs  = 30; warmup = 5
    dummy = torch.randn(1, 1, img_size, img_size, device=device)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        for _ in range(warmup): _ = model(dummy)
        torch.cuda.synchronize()
        s = torch.cuda.Event(enable_timing=True)
        e = torch.cuda.Event(enable_timing=True)
        s.record()
        for _ in range(runs): _ = model(dummy)
        e.record(); torch.cuda.synchronize()
        latency_ms = s.elapsed_time(e) / runs
        max_mem_mb = torch.cuda.max_memory_allocated() / (1024**2)
    else:
        for _ in range(warmup): _ = model(dummy)
        t0 = time.perf_counter()
        for _ in range(runs): _ = model(dummy)
        latency_ms = ((time.perf_counter()-t0)/runs)*1000
        max_mem_mb = float("nan")
    return {"params_m": float(params_m), "gflops": float(gflops),
            "latency_ms": float(latency_ms), "max_mem_mb": float(max_mem_mb)}


print("measure_efficiency ready.")

In [ ]:
def train_one_epoch_baseline(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss, n = 0.0, 0
    agg = {k: 0.0 for k in ["dice","iou","accuracy","recall","precision"]}
    for batch in loader:
        imgs   = batch["image"].to(device, non_blocking=True)
        masks  = batch["mask"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(AMP_DEVICE_TYPE, enabled=USE_AMP):
            out  = model(imgs)
            logits = out[0] if isinstance(out, tuple) else out
            loss = criterion(logits, masks)
        if math.isnan(loss.item()) or math.isinf(loss.item()):
            optimizer.zero_grad(set_to_none=True); continue
        if USE_AMP and scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        bs = imgs.size(0)
        total_loss += loss.item() * bs
        with torch.no_grad():
            m = segmentation_metrics(logits.detach(), masks)
        for k in agg: agg[k] += m[k] * bs
        n += bs
    return total_loss / max(1, n), {k: agg[k] / max(1, n) for k in agg}


@torch.no_grad()
def evaluate_model_baseline(model, loader, criterion):
    model.eval()
    total_loss, n = 0.0, 0
    agg = {k: 0.0 for k in ["dice","iou","accuracy","recall","precision"]}
    imt_abs_err = []
    for batch in loader:
        imgs   = batch["image"].to(device, non_blocking=True)
        masks  = batch["mask"].to(device, non_blocking=True)
        names  = batch["name"]
        roi_sq = batch["roi_square_side"]
        with torch.amp.autocast(AMP_DEVICE_TYPE, enabled=USE_AMP):
            out    = model(imgs)
            logits = out[0] if isinstance(out, tuple) else out
            loss   = criterion(logits, masks)
        bs = imgs.size(0)
        total_loss += loss.item() * bs
        m = segmentation_metrics(logits, masks)
        for k in agg: agg[k] += m[k] * bs
        preds = torch.argmax(logits, dim=1).cpu().numpy().astype(np.float32)
        gts   = masks.cpu().numpy().astype(np.float32)
        for i in range(bs):
            cf = load_calibration_factor(names[i])
            if cf is None: continue
            imt_pred = measure_imt_from_mask(preds[i], cf, float(roi_sq[i]), IMG_SIZE)
            imt_gt   = measure_imt_from_mask(gts[i],   cf, float(roi_sq[i]), IMG_SIZE)
            imt_abs_err.append(abs(imt_pred - imt_gt))
        n += bs
    mae = float(np.mean(imt_abs_err)) if imt_abs_err else float("nan")
    mse = float(np.mean([e**2 for e in imt_abs_err])) if imt_abs_err else float("nan")
    return total_loss / max(1, n), {k: agg[k] / max(1, n) for k in agg}, mae, mse

print("train_one_epoch_baseline / evaluate_model_baseline ready.")


In [ ]:
# ================================================================
# SIMPAN prediksi per-citra pada PUSAT yang di-hold-out (LOCO). CSV-only.
# ================================================================
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

@torch.no_grad()
def _export_csv(model, loader, out_dir, tag):
    model.eval(); rows=[]
    for batch in loader:
        imgs  = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].numpy().astype(np.float32)
        names = batch["name"]; roi = batch["roi_square_side"]
        with torch.amp.autocast(AMP_DEVICE_TYPE, enabled=USE_AMP):
            out = model(imgs); logits = out[0] if isinstance(out, tuple) else out
        pred = (torch.softmax(logits.float(), 1)[:,1].cpu().numpy() > 0.5).astype(np.float32)
        for i in range(len(names)):
            nm=names[i]; p=pred[i]>0.5; g=masks[i]>0.5
            tp=float((p&g).sum()); fp=float((p&~g).sum()); fn=float((~p&g).sum()); eps=1e-6
            cf=load_calibration_factor(nm)
            cp=measure_imt_from_mask(pred[i],  cf, float(roi[i]), IMG_SIZE) if cf else np.nan
            cg=measure_imt_from_mask(masks[i], cf, float(roi[i]), IMG_SIZE) if cf else np.nan
            mm=META.get(nm, {})
            rows.append(dict(name=nm, release=mm.get("release"), center=mm.get("center"),
                snr=mm.get("snr"), morph=mm.get("morph"),
                dice=(2*tp+eps)/(2*tp+fp+fn+eps), iou=(tp+eps)/(tp+fp+fn+eps),
                recall=(tp+eps)/(tp+fn+eps), precision=(tp+eps)/(tp+fp+eps),
                cimt_pred_mm=cp, cimt_gt_mm=cg, imt_abs_err_mm=(abs(cp-cg) if cf else np.nan),
                roi_square_side=float(roi[i]), cf=cf))
    pd.DataFrame(rows).to_csv(out_dir / f"{tag}_per_image.csv", index=False)
    return len(rows)

def save_loco_predictions(model, test_names_fold, fold_name, out_dir):
    ds = CUBSCombinedDataset(test_names_fold, IMG_SIZE, train=False)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                    pin_memory=PIN_MEMORY, persistent_workers=(NUM_WORKERS>0))
    n = _export_csv(model, dl, out_dir, f"loco_{fold_name}_test")
    print(f"  [save loco] {fold_name}: test citra={n} -> {out_dir}/loco_{fold_name}_test_per_image.csv")

print("save_loco_predictions ready.")


In [ ]:
def run_loco_training():
    MODEL_NAME = BASELINE_NAMES[0]   # "bc_vmamba_cubs_a1"
    fold_rows, history_rows = [], []
    for cfg in LOCO_FOLDS:
        nm = cfg["name"]
        print("="*70); print(f"LOCO: leave-{nm}-out  (train={len(cfg['train_names'])} "
              f"val={len(cfg['val_names'])} test={len(cfg['test_names'])})"); print("="*70)
        train_dl, val_dl = make_dataloaders(cfg["train_names"], cfg["val_names"])
        model = get_model(MODEL_NAME, num_classes=2).to(device)
        eff = measure_efficiency(model, IMG_SIZE)
        criterion = SimpleLoss().to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scaler = torch.amp.GradScaler(AMP_DEVICE_TYPE, enabled=USE_AMP)
        warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1e-2, end_factor=1.0, total_iters=WARMUP_EPOCHS)
        cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS-WARMUP_EPOCHS, eta_min=ETA_MIN)
        scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine], [WARMUP_EPOCHS])

        best_dice, best_state, patience, best_epoch = 0.0, None, 0, 1
        for epoch in range(1, NUM_EPOCHS+1):
            tr_loss, tr_m = train_one_epoch_baseline(model, train_dl, optimizer, criterion, scaler)
            va_loss, va_m, va_mae, va_mse = evaluate_model_baseline(model, val_dl, criterion)
            scheduler.step()
            print(f"  [{nm}] Ep{epoch:03d}/{NUM_EPOCHS} val dice={va_m['dice']:.4f} iou={va_m['iou']:.4f} mae={va_mae:.4f}mm")
            history_rows.append(dict(fold=nm, epoch=epoch, val_dice=va_m["dice"], val_mae_cimt_mm=va_mae))
            if va_m["dice"] > best_dice:
                best_dice = va_m["dice"]; best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
                patience = 0; best_epoch = epoch
                torch.save({"fold": nm, "epoch": epoch, "model_state_dict": best_state},
                           OUTPUT_DIR / f"loco_{nm}_best.pth")
            else:
                patience += 1
            if patience >= EARLY_STOP_PAT:
                print(f"  [EarlyStop {nm}] best={best_dice:.4f} @ ep{best_epoch}"); break
            if device.type == "cuda": torch.cuda.empty_cache()

        model.load_state_dict(best_state)
        # evaluasi pada PUSAT yang di-hold-out (generalisasi)
        test_ds = CUBSCombinedDataset(cfg["test_names"], IMG_SIZE, train=False)
        test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                             pin_memory=PIN_MEMORY, persistent_workers=(NUM_WORKERS>0))
        _, te_m, te_mae, te_mse = evaluate_model_baseline(model, test_dl, criterion)
        save_loco_predictions(model, cfg["test_names"], nm, OUTPUT_DIR)
        print(f"  => LOCO {nm}: TEST(held-out) dice={te_m['dice']:.4f} mae={te_mae:.4f}mm")
        fold_rows.append(FoldResult(
            fold=nm, dice=te_m["dice"], iou=te_m["iou"], accuracy=te_m["accuracy"],
            recall=te_m["recall"], precision=te_m["precision"], mae_cimt_mm=te_mae, mse_cimt_mm=te_mse,
            params_m=eff["params_m"], gflops=eff["gflops"], latency_ms=eff["latency_ms"],
            max_mem_mb=eff["max_mem_mb"], best_epoch=best_epoch))
        if device.type == "cuda": torch.cuda.empty_cache()
    return pd.DataFrame([vars(r) for r in fold_rows]), pd.DataFrame(history_rows)

print("run_loco_training ready.")


In [ ]:
import gc
OUTPUT_DIR = OUTPUT_BASE / "loco_bcvmamba"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

loco_df, loco_hist = run_loco_training()
loco_df.to_csv(OUTPUT_DIR / "loco_fold_metrics.csv", index=False)
loco_hist.to_csv(OUTPUT_DIR / "loco_training_history.csv", index=False)

print("\n" + "="*70)
print("RINGKAS LOCO - TEST = pusat yang di-hold-out (generalisasi lintas-pusat)")
print("="*70)
for _, r in loco_df.iterrows():
    print(f"  leave-{r['fold']:10s}-out : Dice={r['dice']:.4f}  CIMT MAE={r['mae_cimt_mm']:.4f} mm")
print("\n  Referensi in-distribution (E1a 5-fold): Dice~0.845  CIMT MAE~0.124 mm")
print("  Gap generalisasi = penurunan thd referensi in-distribution.")
if device.type == "cuda": torch.cuda.empty_cache()
gc.collect()


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

if not all_hist_dfs:
    print("Jalankan training terlebih dahulu.")
else:
    panel_cfg = [
        ("val_dice",        "Val Dice"),
        ("val_loss",        "Val Loss"),
        ("val_mae_cimt_mm", "Val MAE CIMT (mm)"),
    ]
    for bname, hist_df in all_hist_dfs.items():
        out_dir = OUTPUT_BASE / bname
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        for ax, (col, title) in zip(axes, panel_cfg):
            for fid in sorted(hist_df["fold"].unique()):
                df_f = hist_df[hist_df["fold"] == fid]
                ax.plot(df_f["epoch"], df_f[col], label=f"Fold {fid}", linewidth=1.5)
            if col == "val_dice":
                ax.axhline(0.8608, color="steelblue", linestyle=":", alpha=0.8, linewidth=1.2, label="exp28 ref")
            ax.set_title(f"{bname} — {title}")
            ax.set_xlabel("Epoch"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
        plt.suptitle(f"EXP29 — {bname} | {NUM_EPOCHS}ep AdamW SimpleLoss", fontsize=10)
        plt.tight_layout()
        plot_path = out_dir / f"{bname}_training_curves.png"
        plt.savefig(plot_path, dpi=150); plt.show()
        print(f"Saved: {plot_path}")


## Cara Membaca Hasil

CSV per-fold tersimpan di `OUTPUT_BASE / bc_vmamba_cubs_a1 / bc_vmamba_cubs_a1_fold_metrics.csv`.
Untuk comparison dengan baseline lain dari `baselines_segmentation.ipynb`:

1. Bandingkan kolom `dice` dan `mae_cimt_mm` dengan baseline (unet, transunet, vm_unet_baseline)
   pada fold yang sama (SEED=42, 80/20 split identik).
2. Karena loss + augmentasi + protokol identik, **selisih = kontribusi arsitektur murni**.
3. Untuk loop ke konfigurasi A3/A5 (dengan IMTGradeHead + DBSHL+Speckle loss), buat notebook
   lanjutan yang menambah komponen tersebut on top of model di sini.

## Catatan Memori
- TiFusion (yang berisi VSSBlock) hanya dipakai di level dalam (64x64 dan 32x32) + bottleneck.
- Top-2 levels (256x256, 128x128) memakai ConvBlock standar untuk menjaga VRAM Kaggle T4 (15GB).
- Jika ingin Mamba di semua level (PDF-faithful), turunkan `base_ch` ke 16 atau pakai A100.
